# Final checks: shortcut stress test + source-matched subset
**Why this notebook:** the dataset mixes sources. Most healthy images are 1000×1000 JPEGs, while most
unhealthy images are 256×256 PNGs (probably from Rauf et al. 2019). We test:

**A. Stress test** – the saved fold-1 models from the main run are applied to the same fold-1 validation images after
removing format, resolution and background differences.

**B. Source-matched subset** – train and test ONLY on 256×256 PNG images (one source, one format, one size),
with duplicate groups kept in one fold. If accuracy stays high here, the models genuinely see symptoms.

Set `OUT_DIR` to the folder that contains `resnet18_fold1.pt` and `duplicates_removed.csv`. ~20–30 min on T4.

In [ ]:
import os, io, copy, json, random
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, recall_score
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = os.environ.get("DATA_DIR", "/content/drive/MyDrive/Citrus_Plant_Leaves")
OUT_DIR  = os.environ.get("OUT_DIR",  "/content/drive/MyDrive/citrus_results")   # <- EDIT: folder holding resnet18_fold1.pt
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"
PRETRAINED = os.environ.get("PRETRAINED", "1") == "1"
SEED, IMG, BATCH = 42, 224, (16 if QUICK_TEST else 32)
EPOCHS, PATIENCE, LR = (2, 1, 1e-4) if QUICK_TEST else (20, 5, 1e-4)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

# ---- safety checks: stop early if the main-run files are not where we expect ----
dup_csv = f"{OUT_DIR}/duplicates_removed.csv"
assert os.path.exists(dup_csv), f"Not found: {dup_csv}  -> set OUT_DIR to the main-run results folder"
found = [m for m in ["mobilenet_v2","efficientnet_b0","shufflenet_v2","resnet18","squeezenet1_1"] if os.path.exists(f"{OUT_DIR}/{m}_fold1.pt")]
assert found, f"No *_fold1.pt model files in {OUT_DIR}"
print("Models found:", found)

## 0. Rebuild the exact 991-image list and fold 1 of the main run

In [ ]:
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
class_dirs = {}
for d, _, files in os.walk(DATA_DIR):
    if any(f.lower().endswith(EXTS) for f in files): class_dirs[os.path.basename(d)] = d
CLASSES = sorted(class_dirs, key=lambda c: (0 if c.lower().startswith("healthy") else 1, c))
removed = {(os.path.basename(os.path.dirname(p)), os.path.basename(p)) for p in pd.read_csv(dup_csv)["removed"]}
paths, labels = [], []
for i, c in enumerate(CLASSES):
    for f in sorted(os.listdir(class_dirs[c])):
        if f.lower().endswith(EXTS) and (c, f) not in removed:
            paths.append(os.path.join(class_dirs[c], f)); labels.append(i)
labels = np.array(labels)
info = json.load(open(f"{OUT_DIR}/run_info.json")) if os.path.exists(f"{OUT_DIR}/run_info.json") else {}
print(len(paths), "images", dict(zip(CLASSES, np.bincount(labels))))
if "n_images" in info and not QUICK_TEST:
    assert len(paths) == info["n_images"], f"Image count {len(paths)} != main run {info['n_images']} – check DATA_DIR/OUT_DIR"
fold1_va = list(StratifiedKFold(5 if not QUICK_TEST else 2, shuffle=True, random_state=SEED).split(paths, labels))[0][1]

## Shared helpers

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
eval_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
train_tf = transforms.Compose([transforms.RandomResizedCrop(IMG, scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(), transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(s, items, y, tf): s.items, s.y, s.tf = items, y, tf
    def __len__(s): return len(s.items)
    def __getitem__(s, i):
        im = s.items[i] if isinstance(s.items[i], Image.Image) else Image.open(s.items[i]).convert("RGB")
        return s.tf(im), int(s.y[i])

def build_model(name):
    W = "DEFAULT" if PRETRAINED else None
    if name == "shufflenet_v2":   m = models.shufflenet_v2_x1_0(weights=W); m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "resnet18":      m = models.resnet18(weights=W); m.fc = nn.Linear(m.fc.in_features, 2)
    elif name == "mobilenet_v2":  m = models.mobilenet_v2(weights=W); m.classifier[1] = nn.Linear(m.last_channel, 2)
    elif name == "efficientnet_b0": m = models.efficientnet_b0(weights=W); m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    elif name == "squeezenet1_1": m = models.squeezenet1_1(weights=W); m.classifier[1] = nn.Conv2d(512, 2, 1)
    return m

def predict(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            ps.append(torch.softmax(model(x.to(DEVICE)), 1)[:, 1].cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

def leaf_mask(im):
    s = np.asarray(im.convert("HSV"))[:, :, 1].astype(np.uint8)
    hist = np.bincount(s.ravel(), minlength=256).astype(float); w = np.cumsum(hist); mu = np.cumsum(hist * np.arange(256))
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mu[-1] * w / w[-1] - mu) ** 2 / (w * (w[-1] - w))
    return s > int(np.nanargmax(between[:-1]))

def to_jpeg(im, q=90):
    b = io.BytesIO(); im.convert("RGB").save(b, "JPEG", quality=q); b.seek(0); return Image.open(b).convert("RGB")
def grey_bg(im):
    a = np.asarray(im.convert("RGB")).copy(); a[~leaf_mask(im)] = (128, 128, 128); return Image.fromarray(a)
def to_256(im): return im.convert("RGB").resize((256, 256), Image.BILINEAR)

## A. Stress test on the fold-1 models (same validation images as the main run)
`harmonised` = every image resized to 256×256, saved as JPEG q90, background set to grey – this removes
format, resolution and background differences at once.

In [ ]:
VARIANTS = {"original": lambda im: im,
            "jpeg_q90": to_jpeg,
            "resized_256": to_256,
            "grey_background": grey_bg,
            "harmonised": lambda im: grey_bg(to_jpeg(to_256(im)))}
base_imgs = [Image.open(paths[i]).convert("RGB") for i in fold1_va]
yv = labels[fold1_va]
rows = []
for name in found:
    m = build_model(name); m.load_state_dict(torch.load(f"{OUT_DIR}/{name}_fold1.pt", map_location="cpu")); m.to(DEVICE)
    row = {"model": name}
    for vname, fn in VARIANTS.items():
        _, p = predict(m, DataLoader(DS([fn(im) for im in base_imgs], yv, eval_tf), BATCH))
        yhat = (p >= .5).astype(int)
        row[f"{vname}_acc"] = accuracy_score(yv, yhat) * 100
        row[f"{vname}_healthy_recall"] = recall_score(yv, yhat, pos_label=0, zero_division=0) * 100
    rows.append(row); print({k: (round(v, 2) if isinstance(v, float) else v) for k, v in row.items()})
stress = pd.DataFrame(rows).round(2); stress.to_csv(f"{OUT_DIR}/final_A_stress_test.csv", index=False)
stress[["model"] + [c for c in stress.columns if c.endswith("_acc")]]

## B. Source-matched subset: 256×256 PNG images only, duplicate groups kept together
Healthy images are few here, so we report **balanced accuracy** and healthy-class recall, not just accuracy.

In [ ]:
def meta(p):
    with Image.open(p) as im: return (im.format or "").upper(), im.size
sub = [i for i, p in enumerate(paths) if meta(p) == ("PNG", (256, 256))]
sp, sy = [paths[i] for i in sub], labels[sub]
print("Source-matched subset:", len(sp), "images", dict(zip(CLASSES, np.bincount(sy, minlength=2))))
json.dump({"n": len(sp), "per_class": dict(zip(CLASSES, map(int, np.bincount(sy, minlength=2))))},
          open(f"{OUT_DIR}/final_B_subset_counts.json", "w"), indent=2)

# duplicate groups inside the subset (rotation/flip-invariant ResNet-18 similarity >= 0.95, same class)
fe = models.resnet18(weights="DEFAULT" if PRETRAINED else None); fe.fc = nn.Identity(); fe.eval().to(DEVICE)
def dihedral(im):
    return [(ImageOps.mirror(im) if f else im).rotate(r, expand=True) for f in (False, True) for r in (0, 90, 180, 270)]
with torch.no_grad():
    F = np.stack([nn.functional.normalize(fe(torch.stack([eval_tf(v) for v in dihedral(Image.open(p).convert("RGB"))]).to(DEVICE)), dim=1).cpu().numpy() for p in sp])
S = np.max(np.einsum("id,jkd->ijk", F[:, 0], F), axis=2); S = np.maximum(S, S.T); np.fill_diagonal(S, 0)
SIM_T = float(os.environ.get("SIM_T", 0.95))
a, b = np.where(np.triu(S) >= SIM_T); keep = sy[a] == sy[b]
_, g = connected_components(coo_matrix((np.ones(keep.sum()), (a[keep], b[keep])), shape=(len(sp), len(sp))), directed=False)

n_splits = 5 if not QUICK_TEST else 2
n_splits = min(n_splits, np.bincount(sy, minlength=2).min())
assert n_splits >= 2, "Too few healthy (or unhealthy) images in the source-matched subset to run part B" 
cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
res = []
for name in ["shufflenet_v2", "resnet18"]:
    for k, (tr, va) in enumerate(cv.split(sp, sy, g)):
        if len(set(sy[tr])) < 2 or len(set(sy[va])) < 2:
            print(f"fold {k+1}: one class missing after grouping - skipped"); continue
        set_seed(SEED + k)
        tr_dl = DataLoader(DS([sp[i] for i in tr], sy[tr], train_tf), BATCH, shuffle=True, num_workers=0 if QUICK_TEST else 2)
        va_dl = DataLoader(DS([sp[i] for i in va], sy[va], eval_tf), BATCH, num_workers=0 if QUICK_TEST else 2)
        m = build_model(name).to(DEVICE)
        w = torch.tensor(len(tr) / (2 * np.bincount(sy[tr], minlength=2)), dtype=torch.float32, device=DEVICE)
        crit, opt = nn.CrossEntropyLoss(weight=w), torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
        best, state, wait = 1e9, copy.deepcopy(m.state_dict()), 0
        for ep in range(EPOCHS):
            m.train()
            for x, y in tr_dl:
                opt.zero_grad(); l = crit(m(x.to(DEVICE)), y.to(DEVICE)); l.backward(); opt.step()
            m.eval(); vl = 0
            with torch.no_grad():
                for x, y in va_dl: vl += crit(m(x.to(DEVICE)), y.to(DEVICE)).item() * len(y)
            vl /= len(va)
            if vl < best: best, state, wait = vl, copy.deepcopy(m.state_dict()), 0
            else:
                wait += 1
                if wait >= PATIENCE: break
        m.load_state_dict(state); y, p = predict(m, va_dl); yhat = (p >= .5).astype(int)
        r = {"model": name, "fold": k + 1, "n_val": len(va), "healthy_in_val": int((y == 0).sum()),
             "accuracy": accuracy_score(y, yhat), "balanced_accuracy": balanced_accuracy_score(y, yhat),
             "healthy_recall": recall_score(y, yhat, pos_label=0, zero_division=0),
             "unhealthy_recall": recall_score(y, yhat, pos_label=1, zero_division=0),
             "auc": roc_auc_score(y, p) if len(set(y)) > 1 else np.nan}
        res.append(r); print({k2: (round(v, 4) if isinstance(v, float) else v) for k2, v in r.items()})
assert len(res), "No fold could be evaluated in part B"
res = pd.DataFrame(res); res.to_csv(f"{OUT_DIR}/final_B_source_matched_per_fold.csv", index=False)
summ = res.groupby("model")[["accuracy", "balanced_accuracy", "healthy_recall", "unhealthy_recall", "auc"]].agg(["mean", "std"])
summ.to_csv(f"{OUT_DIR}/final_B_source_matched_summary.csv"); print(summ)
print("\nSend back: final_A_stress_test.csv, final_B_subset_counts.json, final_B_source_matched_summary.csv, final_B_source_matched_per_fold.csv")